In [29]:
import rasterio 
import numpy as np
from sklearn.cluster import KMeans
import geopandas as gpd
from shapely.geometry import Point

# ocitavanje rastera i dohvatanje podataka
raster = rasterio.open(r"C:\Users\Svetozar Perisin\GIS Projekte\fruskac_final.tif")
data = raster.read(1)

# filtriranje no data podataka i stavljanje thresholda
mask = (data != raster.nodata) & (data < -70)
rows, cols = np.where(mask)
values = data[rows, cols]
weights = np.abs(values)

# pretvaranje u geokordinate
xs, ys = rasterio.transform.xy(raster.transform, rows, cols)
coords = np.column_stack((xs, ys))

# primena kMeans alogritma
kmeans = KMeans(n_clusters=20, random_state=42).fit(coords, sample_weight=weights)
#kmeans.fit(coords)
labels = kmeans.labels_

#geodata frame pravljenje 
points = [Point(x, y) for x, y in coords]

gdf = gpd.GeoDataFrame(
    {"cluster": labels},
    geometry=points,
    crs=raster.crs
)

#ukloni noise 
gdf = gdf[gdf["cluster"] >= 0]

#napravi zone 
zones = gdf.dissolve(by="cluster")
zones["geometry"] = zones.convex_hull

#export 
zones.to_file("kmeans_hotspots20.geojson", driver="GeoJSON")

print("✅ Gotovo! GeoJSON napravljen.")


✅ Gotovo! GeoJSON napravljen.
